<a href="https://colab.research.google.com/github/RafsanJany-44/ARC/blob/master/LIME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

data = 'https://raw.githubusercontent.com/RafsanJany-44/Thesis_Project/main/All_DATA/without_sn_and_Epoch/EEG_HMC.csv'
dataset = pd.read_csv(data)

from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
dataset["Sleep Stage"]=encoder.fit_transform(dataset["Sleep Stage"])

X = dataset.iloc[:, 1:]
y = dataset.iloc[:, 0]

dic_target = {
    0:"N1"
    1:"N2"
    2:"N3"
    3:"R"
    4:"W"
}

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0)

In [ ]:
X_train.shape, X_test.shape

StandardScaler() returns a numpy array. For better understandings we convert that in to pandas dataframe. StandardScaler() returns a numpy array. For better understanding we convert that in to pandas dataframe. And in separation for X and y, we remove the <b><i>[.values]</i></b>.

In [ ]:
X_train2=X_train
X_test2=X_test

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train =sc.fit_transform(X_train)

X_test =sc.fit_transform(X_test)


X_train = pd.DataFrame(X_train, index=X_train2.index, columns=X_train2.columns)
X_test = pd.DataFrame(X_test, index=X_test2.index, columns=X_test2.columns)


**Tuning**

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
KNN_class = KNeighborsClassifier(n_neighbors=7)
KNN_class.fit(X_train, y_train)
y_pred_KNN=KNN_class.predict(X_test)

Radomforest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=125, random_state=0)
rf.fit(X_train, y_train)
y_pred_RF=rf.predict(X_test)

xgboost

In [ ]:
import xgboost as xgb
xgb_classifier = xgb.XGBClassifier(max_depth=14)
xgb_classifier.fit(X_train.values, y_train.values)  
y_pred_XG = xgb_classifier.predict(X_test.values)

Accurecy

In [ ]:
from sklearn.metrics import accuracy_score
print("Accurecy KNN: ",accuracy_score(y_test, y_pred_KNN))
print("Accurecy RF: ",accuracy_score(y_test, y_pred_RF))
print("Accurecy XG: ",accuracy_score(y_test, y_pred_XG))

#LIME

For further usage and Rearch perpose

In [ ]:
!pip install lime

In [ ]:
X_test.iloc[8]

In [ ]:
ex_specie = np.array(X_test.iloc[10]).reshape(1,-1)

In [ ]:
print(X_test.iloc[[10]])

In [ ]:
print("Prediction for KNN: ",KNN_class.predict(ex_specie)[0])
print("Prediction for RF: ",rf.predict(ex_specie)[0])
print("Prediction for XG: ",xgb_classifier.predict(np.array(X_train.iloc[[10]]))[0])
print("Prediction Accual: ",list(y_test.iloc[[10]])[0])

In [ ]:
print("Prediction Accual: ",list(y_test.iloc[[10]])[0])

In [ ]:
import lime
from lime import lime_tabular
explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=list(dataset.iloc[:0,1:]),
    class_names=['N1','N2','N3','R','W'],
    mode='classification'
)

In [ ]:
exp_KNN = explainer.explain_instance(X_test.iloc[10],
                                 KNN_class.predict_proba,
                                 num_features=6,
                                 top_labels=5)

#exp_KNN.show_in_notebook(show_table=True, show_all=False)

In [ ]:
exp_KNN.save_to_file("KNN.html")

In [ ]:
exp_rf = explainer.explain_instance(X_test.iloc[10],
                                 rf.predict_proba,
                                 num_features=6,
                                 top_labels=5)

#exp_rf.show_in_notebook(show_table=True, show_all=False)

In [ ]:
exp_rf.save_to_file("RandomForest.html")

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values,
                                                   feature_names=list(dataset.iloc[:0,1:]),
                                                   class_names=['N1','N2','N3','R','W'])

In [ ]:
exp_xg = explainer.explain_instance(X_train.iloc[10, :].values,
                                 xgb_classifier.predict_proba,
                                 num_features=6,
                                 top_labels=5)

#exp_xg.show_in_notebook(show_table=True, show_all=False)

In [ ]:
exp_xg.save_to_file("XG_boost.html")



---





---



---



#Reaserch Execution
For only this reasearch

Randomly chossing Rows

In [ ]:
dic_target = {
    0:"N1"
    1:"N2"
    2:"N3"
    3:"R"
    4:"W"
}

In [ ]:
from sklearn.utils.extmath import row_norms
from tqdm.notebook import tqdm
import random
all=[]
none=[]
onlyRF=[]
onlyXG=[]
onlyKNN=[]

RFKNN=[]
RFXG=[]
KNNXG=[]


for i in tqdm(range(0,1000)):
  ex_specie = np.array(X_test.iloc[i]).reshape(1,-1)
  knn=KNN_class.predict(ex_specie)[0]
  ranf=rf.predict(ex_specie)[0]
  xg=xgb_classifier.predict(np.array(X_train.iloc[[i]]))[0]
  y=y_test.iloc[i]
  
  if y==knn and y==ranf and y==xg:
    all.append(i)
  
  elif y==knn and y!=ranf and y!=xg:
    onlyKNN.append(i)
  
  elif y!=knn and y==ranf and y!=xg:
    onlyRF.append(i)
  
  elif y!=knn and y!= ranf and y==xg:
    onlyXG.append(i)
  
  elif y==knn and y==ranf and y!=xg:
    RFKNN.append(i)
  
  elif y!=knn and y==ranf and y==xg:
    RFXG.append(i)
  
  elif y==knn and y!=ranf and y==xg:
    KNNXG.append(i)
  
  else:
    none.append(i)

temp_rows=[all,none,onlyRF,onlyXG,onlyKNN,RFKNN,RFXG,KNNXG]
sec=["ALL","NONE","ONLY_RF","ONLY_XG","ONLY_KNN","RF_KNN","RF_XG","KNN_XG"]
rows=[]

for i in temp_rows:
  rows.append(random.choice(i))

In [ ]:
!pip install lime

In [ ]:
from tqdm import tqdm
import lime
from lime import lime_tabular
import os
feat_num=5


def lime(rows):

  for row in tqdm(rows):
    ex_specie = np.array(X_test.iloc[row]).reshape(1,-1)

    cls_name=['N1','N2','N3','R','W']


    explainer_1 = lime_tabular.LimeTabularExplainer(
        training_data=np.array(X_train),
        feature_names=list(dataset.iloc[:0,1:]),
        class_names=cls_name,
        mode='classification'
    )


    explainer_2 = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=list(dataset.iloc[:0,1:]),
        class_names=cls_name,
    )


    exp_KNN = explainer_1.explain_instance(X_test.iloc[row],
                                    KNN_class.predict_proba,
                                    num_features=feat_num,
                                    top_labels=len(cls_name,))



    exp_rf = explainer_1.explain_instance(X_test.iloc[row],
                                    rf.predict_proba,
                                    num_features=feat_num,
                                    top_labels=len(cls_name,))



    exp_xg = explainer_2.explain_instance(X_train.iloc[row, :].values,
                                    xgb_classifier.predict_proba,
                                    num_features=feat_num,
                                    top_labels=len(cls_name,))

    os.mkdir()
    exp_KNN.save_to_file('/content/ResultHTML/'+sec[rows.index(row)]+"___"+str(row)+"___KNN_.html")
    exp_rf.save_to_file('/content/ResultHTML/'+sec[rows.index(row)]+"___"+str(row)+"___RandomForest_.html")
    exp_xg.save_to_file('/content/ResultHTML/'+sec[rows.index(row)]+"___"+str(row)+"___XG_boost_.html")



In [ ]:
print(rows)
print(sec)

In [ ]:
print(sec[rows.index(49)])